# 05 — Actions Training Pairs

Generates instruction/output training pairs from `Examples/list-of-actions.txt`.
Each single-line example in that file becomes multiple training pairs:

1. **Usage** — "Show me how to use `<verb>` in ARO" → example line
2. **Explanation** — "What does this ARO statement do?" → natural-language description
3. **Context** — "Write a complete feature set that uses `<verb>`" → full ARO block
4. **Variation** — "Write a different example using `<verb>`" → model-generated variation

This covers the action vocabulary exhaustively and anchors the model to correct
preposition usage, qualifier patterns, and result-binding conventions.

**Input:**  `../../Examples/list-of-actions.txt`
            `../data/02_knowledge/knowledge.json`
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl` (appended)
            `../data/04b_actions/actions_pairs.jsonl` (own copy)

In [1]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json, re, sys, random, subprocess
from pathlib import Path

ACTIONS_FILE = ARO_ROOT / 'Examples' / 'list-of-actions.txt'
DATA_OUT     = GLOBAL_OUT_DIR / '../data/04b_actions'
DATA_OUT.mkdir(parents=True, exist_ok=True)

OWN_FILE   = DATA_OUT / 'actions_pairs.jsonl'

with open(DATA_IN / 'knowledge.json') as f:
    kb = json.load(f)

print(f'Actions file: {ACTIONS_FILE}')
print(f'Knowledge base: {len(kb["actions"])} actions')

Config loaded | model=mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
ARO root:  /Users/kris/Projects/ARO-App
Pairs:     /Volumes/Models/data/../data/02_knowledge/knowledge_pairs.jsonl
Adapters:  /Volumes/Models/data/../data/adapters
Actions file: /Users/kris/Projects/ARO-App/Examples/list-of-actions.txt
Knowledge base: 19 actions


## Parse list-of-actions.txt

In [2]:
def parse_actions_file(path):
    """
    Parses list-of-actions.txt into a list of dicts:
      { verb, role, example_line, category }
    Sections are delimited by lines containing '---'.
    """
    content = Path(path).read_text()
    lines   = content.splitlines()

    actions     = []
    current_role = 'UNKNOWN'
    role_re      = re.compile(r'^(\w+) actions', re.IGNORECASE)
    # Match lines like: Extract the <id> from the <pathParameters: id>.
    stmt_re      = re.compile(r'^([A-Z][a-zA-Z]+)\s+(?:the|a|an)?\s*<')

    for line in lines:
        stripped = line.strip()
        # Section header — detect role
        role_m = role_re.match(stripped)
        if role_m:
            current_role = role_m.group(1).upper()
            continue
        # Skip separators and headers
        if not stripped or stripped.startswith('ARO Action') or set(stripped) == {'-'}:
            continue
        # Match statement lines
        stmt_m = stmt_re.match(stripped)
        if stmt_m:
            verb = stmt_m.group(1).lower()
            actions.append({
                'verb':         verb,
                'role':         current_role,
                'example_line': stripped,
            })

    return actions

raw_actions = parse_actions_file(ACTIONS_FILE)
print(f'Parsed {len(raw_actions)} action examples')
for a in raw_actions[:5]:
    print(f'  [{a["role"]:10s}] {a["verb"]:12s} → {a["example_line"][:60]}')

Parsed 58 action examples
  [REQUEST   ] extract      → Extract the <id> from the <pathParameters: id>.
  [REQUEST   ] retrieve     → Retrieve the <user> from the <user-repository> where id = <u
  [REQUEST   ] receive      → Receive the <packet> from the <socket: message>.
  [REQUEST   ] read         → Read the <content> from the <config-file>.
  [REQUEST   ] request      → Request the <response> from the <api-url>.


## Build action metadata index

Merge parsed examples with metadata from `knowledge.json` (verbs, prepositions, description).

In [3]:
# Build a lookup: primary_verb -> action metadata
action_meta = {}
for a in kb.get('actions', []):
    for v in a.get('verbs', []):
        action_meta[v.lower()] = a

# Enrich parsed actions with metadata
for a in raw_actions:
    meta = action_meta.get(a['verb'], {})
    a['all_verbs']     = meta.get('verbs', [a['verb']])
    a['prepositions']  = meta.get('prepositions', [])
    a['description']   = meta.get('description', '')
    a['action_name']   = meta.get('name', a['verb'].capitalize() + 'Action')

print(f'Enriched {len(raw_actions)} actions with metadata')
print(f'Actions with description: {sum(1 for a in raw_actions if a["description"])}')

Enriched 58 actions with metadata
Actions with description: 18


## Load model

In [4]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

model, tokenizer, _, mlx_generate, make_sampler = load_model()

Config loaded | model=mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
ARO root:  /Users/kris/Projects/ARO-App
Pairs:     /Volumes/Models/data/../data/02_knowledge/knowledge_pairs.jsonl
Adapters:  /Volumes/Models/data/../data/adapters


/Library/Python/3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/kris/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit with warm-start adapter...


Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 178734.55it/s]


  Adapter: /Volumes/Models/data/../data/adapters/warm_start
Model ready.


## System prompt

In [5]:
syntax_summary = kb.get('aro_syntax', '')[:2000]

SYSTEM_PROMPT = f"""You are an expert ARO (Action Result Object) programmer and teacher.

ARO syntax: Verb the <Result> preposition [the] <Object>.
Every statement is either:
  Verb the <Result: qualifier> from/to/with/for/on/into the <Object: qualifier>.
  Verb <literal> to the <Object>.
  Publish as <alias> <variable>.

{syntax_summary}

Rules:
- Variables are immutable — use new names for each transform
- String concat: ++ (not +)
- Feature set: (Name: Business Activity) {{ statements }}
- Exactly one Application-Start per application
- Return an <OK: status> ... to end a feature set
- Qualifiers after : in angle brackets specify field/operation"""

print(f'System prompt: {len(SYSTEM_PROMPT)} chars')

System prompt: 696 chars


## Pair templates

For each action example line, generate 4 types of training pairs.

In [6]:
def make_static_pairs(action):
    """
    Returns static (non-generated) pairs that don't require the LLM.
    These are high-quality, deterministic pairs.
    """
    verb   = action['verb']
    line   = action['example_line']
    role   = action['role']
    verbs  = action['all_verbs']
    preps  = action['prepositions']
    name   = action['action_name']

    verb_list  = ', '.join(f'`{v}`' for v in verbs[:4])
    prep_list  = ', '.join(f'`{p}`' for p in preps[:4]) if preps else 'none'

    pairs = []

    # 1. Direct usage question
    pairs.append({
        'instruction': f'Show me how to use the `{verb}` action in ARO.',
        'output':      line,
        'source':      'actions_usage',
        'score':       1.0,
    })

    # 2. Verb alias questions
    for alias in verbs[1:3]:  # up to 2 aliases
        pairs.append({
            'instruction': f'How do I use `{alias}` in ARO? (alias for {verb})',
            'output':      line.replace(verb.capitalize(), alias.capitalize(), 1),
            'source':      'actions_alias',
            'score':       0.9,
        })

    # 3. What does this line do?
    pairs.append({
        'instruction': f'What does this ARO statement do?\n\n{line}',
        'output':      (
            f'This statement uses the `{verb}` action ({role} role).\n'
            f'Available verbs: {verb_list}\n'
            f'Valid prepositions: {prep_list}\n\n'
            f'The result is bound to the variable in the first angle bracket, '
            f'and the source is taken from the last angle bracket.'
        ),
        'source':      'actions_explain',
        'score':       1.0,
    })

    # 4. Which action to use?
    role_desc = {
        'REQUEST':  'pulls data from an external source into the feature set',
        'OWN':      'transforms or computes data within the feature set',
        'RESPONSE': 'returns a result or error to the caller',
        'EXPORT':   'persists, broadcasts, or publishes data',
        'FILE':     'performs a file system operation',
        'TERMINAL': 'interacts with the terminal display',
        'SERVICE':  'manages a long-running service',
        'TEST':     'performs a test assertion',
        'SCHEDULE': 'schedules a recurring event',
    }.get(role, 'performs an operation')

    pairs.append({
        'instruction': f'In ARO, which action {role_desc} using the verb `{verb}`? Give an example.',
        'output':      f'The `{verb}` action ({name}).\n\nExample:\n{line}',
        'source':      'actions_which',
        'score':       1.0,
    })

    return pairs


# Test
sample = raw_actions[0]
static_pairs = make_static_pairs(sample)
print(f'Static pairs for "{sample["verb"]}": {len(static_pairs)}')
for p in static_pairs:
    print(f'  [{p["source"]}] {p["instruction"][:60]}')

Static pairs for "extract": 5
  [actions_usage] Show me how to use the `extract` action in ARO.
  [actions_alias] How do I use `parse` in ARO? (alias for extract)
  [actions_alias] How do I use `get` in ARO? (alias for extract)
  [actions_explain] What does this ARO statement do?

Extract the <id> from the 
  [actions_which] In ARO, which action pulls data from an external source into


## LLM-generated pairs

For each action, ask the model to:
1. Write a complete feature set using the action
2. Write a variation with different domain/qualifiers

In [7]:
MAX_NEW_TOKENS = 600

def generate(prompt, temp=0.5):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    out    = mlx_generate(
        model, tokenizer,
        prompt=tokens,
        max_tokens=MAX_NEW_TOKENS,
        sampler=make_sampler(temp=temp),
        verbose=False,
    )
    # Strip accidental markdown fences
    out = re.sub(r'^```[a-z]*\n?', '', out.strip())
    out = re.sub(r'\n?```$', '', out.strip())
    return out.strip()


CONTEXT_PROMPTS = [
    lambda a: (
        f"Write a complete ARO feature set (Application-Start + one more feature set) "
        f"that demonstrates the `{a['verb']}` action.\n"
        f"Example of the action: `{a['example_line']}`\n"
        f"Use a realistic domain (e.g. user management, e-commerce, file processing)."
    ),
    lambda a: (
        f"Show a different way to use `{a['verb']}` in ARO with a different domain. "
        f"Write just the ARO statement (one line).\n"
        f"Original: `{a['example_line']}`"
    ),
    lambda a: (
        f"Write an ARO feature set that uses `{a['verb']}` together with at least "
        f"two other related actions. Show the full context."
    ),
]

def make_llm_pairs(action):
    pairs = []
    for prompt_fn in CONTEXT_PROMPTS:
        prompt = prompt_fn(action)
        try:
            output = generate(prompt)
            if len(output.strip()) > 20:
                pairs.append({
                    'instruction': prompt,
                    'output':      output,
                    'source':      'actions_context',
                    'score':       0.85,
                })
        except Exception as e:
            print(f'  [LLM] Error for {action["verb"]}: {e}')
    return pairs

print('LLM pair generation ready.')

LLM pair generation ready.


## Main generation loop

In [8]:
try:
    import ipywidgets
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

all_pairs = []

# Resume: load already-generated pairs
already_verbs = set()
if OWN_FILE.exists():
    with open(OWN_FILE) as f:
        for line in f:
            line = line.strip()
            if line:
                p = json.loads(line)
                all_pairs.append(p)
    # Reconstruct which verbs are done from static pairs
    for p in all_pairs:
        m = re.search(r'`(\w+)` action', p.get('instruction', ''))
        if m:
            already_verbs.add(m.group(1).lower())
    print(f'Resuming — {len(all_pairs)} pairs already generated')

outf = open(OWN_FILE, 'a')

try:
    pbar = tqdm(total=len(raw_actions), desc='Actions', unit='action')
    for action in raw_actions:
        verb = action['verb']
        pbar.set_description(f'Actions [{verb}]')

        # ── Static pairs (always regenerate to keep idempotent) ──────────────
        if verb not in already_verbs:
            static = make_static_pairs(action)
            for p in static:
                all_pairs.append(p)
                outf.write(json.dumps(p) + '\n')

            # ── LLM-generated context pairs ──────────────────────────────────
            llm = make_llm_pairs(action)
            for p in llm:
                all_pairs.append(p)
                outf.write(json.dumps(p) + '\n')

            already_verbs.add(verb)
            outf.flush()
            pbar.set_postfix({'total': len(all_pairs), 'verb': verb})

        pbar.update(1)
finally:
    pbar.close()
    outf.close()

print(f'\nTotal action training pairs: {len(all_pairs)}')
from collections import Counter
source_counts = Counter(p['source'] for p in all_pairs)
for src, cnt in sorted(source_counts.items()):
    print(f'  {src:<25} {cnt:4d}')

Actions [assert]: 100%|██████████| 58/58 [18:36<00:00, 19.26s/action, total=361, verb=assert]      


Total action training pairs: 361
  actions_alias               13
  actions_context            174
  actions_explain             58
  actions_usage               58
  actions_which               58


## Merge into main knowledge pairs

In [9]:
existing_keys = set()
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            line = line.strip()
            if line:
                p = json.loads(line)
                existing_keys.add(p.get('instruction', '')[:80])

new_count = 0
with open(PAIRS_FILE, 'a') as f:
    for pair in all_pairs:
        key = pair.get('instruction', '')[:80]
        if key not in existing_keys:
            f.write(json.dumps(pair) + '\n')
            existing_keys.add(key)
            new_count += 1

print(f'Merged {new_count} new action pairs into {PAIRS_FILE}')
print(f'Total pairs now: {len(existing_keys)}')
print()
print('Next steps:')
print('  → Run notebook 04 again to retrain warm-start adapter on enriched pairs')
print('  → Or proceed to notebook 06 (execution-grounded pairs)')

Merged 276 new action pairs into /Volumes/Models/data/../data/02_knowledge/knowledge_pairs.jsonl
Total pairs now: 497

Next steps:
  → Run notebook 04 again to retrain warm-start adapter on enriched pairs
  → Or proceed to notebook 06 (execution-grounded pairs)
